# Marketing brochure for the Battersea Dogs & Cats Home 

## Built by: Madelene Campos

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

Built with ollama's model llama3.2 

In [8]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import json
from IPython.display import Markdown, display, update_display
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

def fetch_website_links(url):
	response = requests.get(url)
	soup = BeautifulSoup(response.text, 'html.parser')
	links = []
	for a in soup.find_all('a', href=True):
		links.append(a['href'])
	return links

def fetch_website_contents(url):
	response = requests.get(url)
	soup = BeautifulSoup(response.text, 'html.parser')
	# Get visible text from the page
	texts = soup.stripped_strings
	return ' '.join(texts)

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
api_key = 'ollama'

if api_key and api_key.startswith('ol'):
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2'

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key)

API key looks good so far


In [10]:
links = fetch_website_links("https://www.battersea.org.uk/")
links

['#main-content',
 'https://www.battersea.org.uk/',
 'https://donate.battersea.org.uk/appeals/default/',
 'https://www.battersea.org.uk/what-we-do',
 'https://www.battersea.org.uk/what-we-do',
 'https://www.battersea.org.uk/rehome',
 'https://www.battersea.org.uk/what-we-do/caring-for-our-animals',
 'https://www.battersea.org.uk/what-we-do/animal-welfare-campaigning',
 'https://www.battersea.org.uk/what-we-do/working-with-other-rescues',
 'https://www.battersea.org.uk/pet-advice',
 'https://www.battersea.org.uk/about-us',
 'https://www.battersea.org.uk/rehome',
 'https://www.battersea.org.uk/rehome',
 'https://www.battersea.org.uk/rehome/dogs',
 'https://www.battersea.org.uk/rehome/cats',
 'https://www.battersea.org.uk/dogs/dog-rehoming-gallery',
 'https://www.battersea.org.uk/cats/cat-rehoming-gallery',
 'https://www.battersea.org.uk/rehome/giving-your-dog-adoption',
 'https://www.battersea.org.uk/rehome/giving-your-cat-adoption',
 'https://www.battersea.org.uk/pet-advice',
 'https://

## First step: Have Ollama figure out which links are relevant

### Use a call to Ollama to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [33]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [34]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do NOT include Terms and conditions, Privacy, Login, Sitemap, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [35]:
print(get_links_user_prompt("https://www.battersea.org.uk/"))


Here is the list of links on the website https://www.battersea.org.uk/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do NOT include Terms and conditions, Privacy, Login, Sitemap, email links.

Links (some might be relative links):

#main-content
https://www.battersea.org.uk/
https://donate.battersea.org.uk/appeals/default/
https://www.battersea.org.uk/what-we-do
https://www.battersea.org.uk/what-we-do
https://www.battersea.org.uk/rehome
https://www.battersea.org.uk/what-we-do/caring-for-our-animals
https://www.battersea.org.uk/what-we-do/animal-welfare-campaigning
https://www.battersea.org.uk/what-we-do/working-with-other-rescues
https://www.battersea.org.uk/pet-advice
https://www.battersea.org.uk/about-us
https://www.battersea.org.uk/rehome
https://www.battersea.org.uk/rehome
https://www.battersea.org.uk/rehome/dogs
https://www.battersea.org.uk/rehome/cats
https://www.battersea.org.uk/dogs/dog-

In [36]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result) # load string
    return links
    

In [37]:
select_relevant_links("https://www.battersea.org.uk/")

{'links': [{'type': 'company/homepage',
   'url': 'https://www.battersea.org.uk/'},
  {'type': 'about us page', 'url': 'https://www.battersea.org.uk/about-us'},
  {'type': 'pet advice/dog care advice',
   'url': 'https://www.battersea.org.uk/pet-advice/dog-care-advice/winter-dog-care'},
  {'type': 'pet advice/cat care advice',
   'url': 'https://www.battersea.org.uk/pet-advice/cat-care-advice/winter-cat-care'},
  {'type': 'rehome page', 'url': 'https://www.battersea.org.uk/rehome'},
  {'type': 'rehome/dogs page',
   'url': 'https://www.battersea.org.uk/rehome/dogs'},
  {'type': 'dogs/dog rehoming gallery',
   'url': 'https://www.battersea.org.uk/dogs/dog-rehoming-gallery'},
  {'type': 'cats/cat rehoming gallery',
   'url': 'https://www.battersea.org.uk/cats/cat-rehoming-gallery'},
  {'type': 'rehome/cats page',
   'url': 'https://www.battersea.org.uk/rehome/cats'},
  {'type': 'rehome/giving your cat adoption',
   'url': 'https://www.battersea.org.uk/rehome/giving-your-cat-adoption'},
 

## Second step: make the brochure!

Assemble all the details into another prompt to Ollama

In [38]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [39]:
print(fetch_page_and_all_relevant_links("https://www.battersea.org.uk/"))

## Landing Page:

Battersea Dogs & Cats Home | Here For Every Dog And Cat Skip to main content Donate Button Donate What we do What we do We've been here for every dog and cat since 1860 Rehoming Caring for our animals Animal welfare campaigning Working with other rescues Supporting pet owners More about us Rehoming Rehoming Finding new homes for dogs and cats How to rehome a dog How to rehome a cat Meet the dogs Meet the cats Giving your dog up for adoption Giving your cat up for adoption Pet advice Pet advice Sharing our expertise to support pet owners everywhere Dog advice Cat advice Support us Support us Help us be here for every dog and cat Donate Other ways to give Paw Draw weekly lottery Play our raffle Sponsor our kennels or cabins Virtual gifts Challenge events Fundraising Gifts in Wills Donate in memory Volunteer Fostering Business Partnerships Philanthropy Shop Donate Log in Search Keywords Search POPULAR ON BATTERSEA Contacting Battersea Challenge events Pet advice Search C

In [40]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [41]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [42]:
get_brochure_user_prompt("Battersea Dogs And Cats Home", "https://www.battersea.org.uk/")

"\nYou are looking at a company called: Battersea Dogs And Cats Home\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nBattersea Dogs & Cats Home | Here For Every Dog And Cat Skip to main content Donate Button Donate What we do What we do We've been here for every dog and cat since 1860 Rehoming Caring for our animals Animal welfare campaigning Working with other rescues Supporting pet owners More about us Rehoming Rehoming Finding new homes for dogs and cats How to rehome a dog How to rehome a cat Meet the dogs Meet the cats Giving your dog up for adoption Giving your cat up for adoption Pet advice Pet advice Sharing our expertise to support pet owners everywhere Dog advice Cat advice Support us Support us Help us be here for every dog and cat Donate Other ways to give Paw Draw weekly lottery Play our raffle Sponsor our kennels or cabins Virtual g

In [24]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("Battersea Dogs And Cats Home", "https://www.battersea.org.uk/")

Selecting relevant links for https://www.battersea.org.uk/ by calling llama3.2
Found 7 relevant links


**Welcome to Battersea Dogs & Cats Home: Where Love Goes All In**

Are you looking for a place where your furry friends will be loved and cared for? Look no further than Battersea Dogs & Cats Home, one of the UK's oldest and most respected animal welfare organizations. For over 160 years, we have been dedicated to rescuing, rehabilitating, and rehoming dogs and cats in need.

**Our Mission**

We believe that every dog and cat deserves a loving home, regardless of their background or circumstances. Our team works tirelessly to match our animals with the perfect forever families, while also providing expert care and welfare support to those who are already with us.

**Our Work**

At Battersea, we do more than just rescue and care for animals – we campaign for animal welfare, work with other rescues to make a bigger impact, and provide expert advice and guidance to our fellow pet owners. Our efforts have led to the adoption of over 200,000 dogs and cats into loving homes!

**Meet Our Furry Friends**

Looking for some furry inspiration? Browse our profiles of adoptable dogs and cats here at Battersea! From playful pups to snuggly kitties, we're sure you'll find a new best friend among our wonderful bunch.

**Join the Team**

Want to join the Battersea crew? We're always on the lookout for passionate and dedicated individuals to help us continue our vital work. Apply now to become a foster carer, volunteer, or apply for a job with us!

**Shop with Us**

Get exclusive animal-themed gear from our Teemill Store, where every purchase supports our life-saving work.

**Support Us**

Want to make a difference in the lives of dogs and cats everywhere? Play our Winter Raffle today, sponsor our adoption, or leave a gift in your will – every donation counts!

At Battersea Dogs & Cats Home, we're living our promise: "We're all in for them." Join us on this journey and let's go all in for the animals!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [43]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [44]:
stream_brochure("Battersea Dogs And Cats Home", "https://www.battersea.org.uk/")

**Battersea Dogs & Cats Home: Where Every Dog And Cat Deserves A Second Chance**

At the heart of all we do is one simple principle: we believe every dog and cat deserves a loving home, no matter what their story.

For over 160 years, Battersea has been dedicated to helping animals in need. Our journey began with just six dogs and two cats, but our mission has only grown stronger with time. Today, we care for thousands of furry friends every year, providing them with food, shelter, love, and a second chance at life.

**Our Culture**

We're all in, for them. Battersea goes all in for the dogs and cats we support, however they need us and wherever they are. Our team is passionate, dedicated, and works tirelessly to ensure every animal receives the care and attention they deserve. We believe that with great strength comes great love, and we strive to embody this philosophy in everything we do.

**Caring For Animals**

At Battersea, caring for animals is at the heart of what we do. Our expert team provides advice and guidance on everything from feeding and exercise needs to health checks and more. We also work closely with other rescue organizations around the world to help animals beyond our gates.

**Rehoming With Battersea**

Rehoming is a top priority for us, and we strive to match each dog and cat with its perfect forever home. Whether you're looking for a furry friend to join your family or just want to give a loving animal a second chance, we've got you covered.

**Meet Our Pack**

From energetic dogs to sassy felines, our residents are waiting for their forever homes. Take a look at some of our furry friends and see if one of them has caught your eye.

*   **Meet the Dogs**: From playful puppies to gentle seniors, we have a range of breeds and mixes to suit every lifestyle.
*   **Meet the Cats**: Looking for affection? We've got plenty of cat companions waiting to snuggle up in your lap.